<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/QWEN-AIO-NSFW-EDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Cell 1: install ComfyUI + ComfyUI-GGUF and download the Qwen Rapid AIO edit models
# --- Google Drive: keep downloaded LoRAs between Colab sessions (set to False to skip) ---
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")  # Colab asks you to allow access

%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI || true
!cd /content/ComfyUI && git pull -q || true
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q torchsde gguf fastapi uvicorn pydantic requests nest_asyncio

# ComfyUI-GGUF for loading .gguf diffusion models
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-GGUF || git clone -q https://github.com/city96/ComfyUI-GGUF.git)
!cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF && pip install -q -r requirements.txt

!apt install -qqy aria2

!mkdir -p /content/ComfyUI/models/unet
!mkdir -p /content/ComfyUI/models/diffusion_models
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae
!mkdir -p /content/ComfyUI/models/loras

# Store LoRAs in MyDrive/ComfyUI/loras_qwen: ComfyUI's loras folder becomes a link to it, so LoRAs
# downloaded from the website are saved to Drive and show up again in every new session.
import os, shutil
LORA_LOCAL = "/content/ComfyUI/models/loras"
if USE_GOOGLE_DRIVE:
    DRIVE_LORAS = "/content/drive/MyDrive/ComfyUI/loras_qwen"
    os.makedirs(DRIVE_LORAS, exist_ok=True)
    if not os.path.islink(LORA_LOCAL):
        if os.path.isdir(LORA_LOCAL):
            for f in os.listdir(LORA_LOCAL):  # keep anything already downloaded in this session
                src, dst = os.path.join(LORA_LOCAL, f), os.path.join(DRIVE_LORAS, f)
                if not f.startswith("put_") and not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(LORA_LOCAL)
        os.symlink(DRIVE_LORAS, LORA_LOCAL)
    n = len([f for f in os.listdir(DRIVE_LORAS) if not f.endswith(".part")])
    print(f"LoRAs are stored in Google Drive (MyDrive/ComfyUI/loras_qwen): {n} file(s)")
else:
    os.makedirs(LORA_LOCAL, exist_ok=True)
    print("LoRAs are stored on this runtime only (lost when the runtime is deleted)")

# 1. Qwen-Rapid-NSFW-v23 Q2_K GGUF (7.44 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Novice25/Qwen-Image-Edit-Rapid-AIO-GGUF/resolve/main/v23/Qwen-Rapid-NSFW-v23_Q2_K.gguf \
  -d /content/ComfyUI/models/unet -o Qwen-Rapid-NSFW-v23_Q2_K.gguf

# 2. Qwen 2.5 VL 7B text encoder, GGUF Q4_K_M (4.7 GB) + its vision part (mmproj, 1.35 GB).
#    About 6 GB instead of 8.8 GB for the fp8 version, so the text encoder and the diffusion model
#    fit on the T4 together. With the fp8 one, Colab's 12.7 GB of RAM ran out when the two were
#    swapped between GPU and RAM, which crashed the runtime on the second edit.
#    The mmproj file must be named after the text encoder so ComfyUI-GGUF can find it.
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf \
  -d /content/ComfyUI/models/text_encoders -o Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-F16.gguf \
  -d /content/ComfyUI/models/text_encoders -o Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf

# 3. Qwen Image VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors \
  -d /content/ComfyUI/models/vae -o qwen_image_vae.safetensors

!ls -lh /content/ComfyUI/models/unet/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/

/content
fatal: destination path '/content/ComfyUI' already exists and is not an empty directory.
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main.py", line 84, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/__init__.py", line 121, in create_command
    module = importlib.import_module(module_path)
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "<frozen importlib._bootstrap>", line 1404, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1369, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1340, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  Fil

In [2]:
# Cell 2: load the models and define the Qwen image-edit helpers used by cells 3 and 4
%cd /content/ComfyUI

import os
import sys
import gc
import math
import hashlib
import logging
import warnings
import importlib.util

import torch
import numpy as np
from PIL import Image

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

if "/content/ComfyUI" not in sys.path:
    sys.path.insert(0, "/content/ComfyUI")

import folder_paths
import nodes
import node_helpers
import comfy.utils

MODEL_NAME = "Qwen-Rapid-NSFW-v23_Q2_K.gguf"
CLIP_NAME = "Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf"  # vision part: Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf
VAE_NAME = "qwen_image_vae.safetensors"

# Recommended settings for Rapid AIO v23 (distilled: few steps, CFG 1)
AIO_DEFAULTS = {
    "steps": 4,
    "cfg": 1.0,
    "sampler": "sa_solver",
    "scheduler": "beta",
    "width": 1024,
    "height": 1024,
    "denoise": 1.0,
}

# --- Load ComfyUI-GGUF (custom nodes are not auto-loaded when ComfyUI is used as a library)
if "UnetLoaderGGUF" not in nodes.NODE_CLASS_MAPPINGS:
    gguf_path = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/__init__.py"
    spec = importlib.util.spec_from_file_location(
        "ComfyUI-GGUF", gguf_path,
        submodule_search_locations=[os.path.dirname(gguf_path)],
    )
    module = importlib.util.module_from_spec(spec)
    sys.modules["ComfyUI-GGUF"] = module
    spec.loader.exec_module(module)
    nodes.NODE_CLASS_MAPPINGS.update(getattr(module, "NODE_CLASS_MAPPINGS", {}))
    nodes.NODE_DISPLAY_NAME_MAPPINGS.update(getattr(module, "NODE_DISPLAY_NAME_MAPPINGS", {}))

UnetLoaderGGUF = nodes.NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()
CLIPLoaderGGUF = nodes.NODE_CLASS_MAPPINGS["CLIPLoaderGGUF"]()
VAELoader = nodes.NODE_CLASS_MAPPINGS["VAELoader"]()
KSampler = nodes.NODE_CLASS_MAPPINGS["KSampler"]()
VAEEncode = nodes.NODE_CLASS_MAPPINGS["VAEEncode"]()
VAEDecode = nodes.NODE_CLASS_MAPPINGS["VAEDecode"]()
ConditioningZeroOut = nodes.NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()

gc.collect()
torch.cuda.empty_cache()

with torch.inference_mode():
    unet = UnetLoaderGGUF.load_unet(MODEL_NAME)[0]
    clip = CLIPLoaderGGUF.load_clip(CLIP_NAME, "qwen_image")[0]
    vae = VAELoader.load_vae(VAE_NAME)[0]


# --- Qwen Image Edit Plus text encoding.
# TextEncodeQwenImageEditPlus lives in comfy_extras and is NOT in nodes.NODE_CLASS_MAPPINGS
# unless ComfyUI's server initialises the extra nodes, so we reproduce it here with the same
# logic: the image goes to the Qwen2.5-VL vision tower (~384x384) AND is VAE-encoded (~1MP)
# as a reference latent. Without both, the model ignores the input image.
QWEN_EDIT_TEMPLATE = (
    "<|im_start|>system\nDescribe the key features of the input image (color, shape, size, texture, "
    "objects, background), then explain how the user's text instruction should alter or modify the "
    "image. Generate a new image that meets the user's requirements while maintaining consistency "
    "with the original input where appropriate.<|im_end|>\n<|im_start|>user\n{}<|im_end|>\n"
    "<|im_start|>assistant\n"
)


def _resize_to_area(samples, total, multiple=1):
    scale = math.sqrt(total / (samples.shape[3] * samples.shape[2]))
    w = max(multiple, round(samples.shape[3] * scale / multiple) * multiple)
    h = max(multiple, round(samples.shape[2] * scale / multiple) * multiple)
    return comfy.utils.common_upscale(samples, w, h, "area", "disabled")


def qwen_edit_encode(clip, prompt, vae=None, images=(), ref_latents=None):
    """Equivalent of ComfyUI's TextEncodeQwenImageEditPlus. images: list of [1,H,W,3] tensors.
    ref_latents: optional already-encoded reference latents. Passing the output-sized latent
    here (instead of the node's fixed ~1024x1024 re-encode) keeps the model's work proportional
    to the output size, so smaller outputs are actually faster."""
    images = [im for im in images if im is not None][:3]
    images_vl, image_prompt = [], ""
    compute_refs = ref_latents is None
    ref_latents = [] if compute_refs else list(ref_latents)
    for i, image in enumerate(images):
        samples = image[:, :, :, :3].movedim(-1, 1)
        images_vl.append(_resize_to_area(samples, 384 * 384).movedim(1, -1))
        if compute_refs and vae is not None:
            ref = _resize_to_area(samples, 1024 * 1024, 8).movedim(1, -1)
            ref_latents.append(vae.encode(ref))
        image_prompt += "Picture {}: <|vision_start|><|image_pad|><|vision_end|>".format(i + 1)

    if images_vl:
        tokens = clip.tokenize(image_prompt + prompt, images=images_vl, llama_template=QWEN_EDIT_TEMPLATE)
    else:
        tokens = clip.tokenize(prompt)
    cond = clip.encode_from_tokens_scheduled(tokens)
    if ref_latents:
        cond = node_helpers.conditioning_set_values(cond, {"reference_latents": ref_latents}, append=True)
    return cond


def pil_to_comfy(pil_img):
    arr = np.array(pil_img.convert("RGB")).astype(np.float32) / 255.0
    return torch.from_numpy(arr)[None, ...]


def comfy_to_pil(comfy_img, index=0):
    arr = (comfy_img[index].detach().cpu().numpy().clip(0.0, 1.0) * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)


MAX_SIDE = 1536  # hard safety cap for a T4


def output_size(width, height, multiple=16, min_side=256):
    """The Width/Height from the website, rounded to multiples of 16 and kept within 256..MAX_SIDE."""
    w = min(max(int(round(int(width) / multiple)) * multiple, min_side), MAX_SIDE)
    h = min(max(int(round(int(height) / multiple)) * multiple, min_side), MAX_SIDE)
    return w, h


def same_shape_size(src_w, src_h, w, h, multiple=8, max_area=1024 * 1024):
    """Reference-image size: the output frame itself if the shapes (nearly) match, otherwise the
    source's aspect ratio at about w*h pixels, capped at ~1 MP (ComfyUI's Qwen edit default)."""
    if abs((src_w / src_h) / (w / h) - 1) < 0.02 and w * h <= max_area:
        return w, h
    scale = math.sqrt(min(w * h, max_area) / (src_w * src_h))
    return (max(multiple, round(src_w * scale / multiple) * multiple),
            max(multiple, round(src_h * scale / multiple) * multiple))


def center_crop_resize(img, w, h):
    """Resize to cover w x h and crop the middle (no stretching)."""
    scale = max(w / img.width, h / img.height)
    nw, nh = max(w, round(img.width * scale)), max(h, round(img.height * scale))
    img = img.resize((nw, nh), Image.Resampling.LANCZOS)
    left, top = (nw - w) // 2, (nh - h) // 2
    return img.crop((left, top, left + w, top + h))


# Re-rolls with the same prompt + image + size reuse the encoded prompt/image, which skips the
# 7B text encoder and avoids swapping it into VRAM in place of the diffusion model.
_COND_CACHE = {}
_COND_CACHE_MAX = 4


def _cache_key(*parts):
    h = hashlib.sha1()
    for p in parts:
        h.update(p if isinstance(p, bytes) else repr(p).encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()


def run_qwen_edit(positive, negative="", input_pil=None, width=1024, height=1024, steps=4, cfg=1.0,
                  sampler="sa_solver", scheduler="beta", seed=None, denoise=1.0, batch_size=1,
                  on_image=None, on_info=None, model=None, extra_images=()):
    """Edit input_pil (or generate from scratch when None). Returns (list of PIL images, seeds).
    model: the diffusion model to sample with (defaults to the base model; pass one with LoRAs applied).
    extra_images: up to 2 more PIL images ("Picture 2" and "Picture 3" in the prompt) the model can
    take people, clothes, objects or style from. input_pil is "Picture 1" and sets the frame."""
    import random
    # Output is exactly the requested Width x Height (rounded to /16). The input image is never
    # stretched: the model sees it at its own aspect ratio as the reference, and the edit is
    # composed into the requested frame. When the shapes match, this is a plain same-frame edit.
    w, h = output_size(width, height)
    partial = denoise < 1.0  # only then does the starting latent need the input image's content
    if input_pil is not None:
        input_pil = input_pil.convert("RGB")
        rw, rh = same_shape_size(input_pil.width, input_pil.height, w, h)
        ref_pil = input_pil if input_pil.size == (rw, rh) else input_pil.resize((rw, rh), Image.Resampling.LANCZOS)
        ref_tensor = pil_to_comfy(ref_pil)
        image_bytes = ref_pil.tobytes()
    else:
        ref_tensor = None
        image_bytes = b""

    # Pictures 2 and 3: own aspect ratio, kept smaller (~0.6 MP max) because every reference
    # image adds work to every sampling step.
    extra_tensors = []
    for extra in [im for im in (extra_images or []) if im is not None][: (2 if input_pil is not None else 3)]:
        extra = extra.convert("RGB")
        ew, eh = same_shape_size(extra.width, extra.height, w, h, max_area=768 * 768)
        extra = extra if extra.size == (ew, eh) else extra.resize((ew, eh), Image.Resampling.LANCZOS)
        extra_tensors.append(pil_to_comfy(extra))
        image_bytes += b"|" + extra.tobytes()

    use_neg = cfg != 1.0  # negative is ignored at CFG 1, so skip encoding it
    key = _cache_key(positive, negative if use_neg else None, w, h, image_bytes, partial)
    cached = key in _COND_CACHE
    if on_info:
        on_info({"width": w, "height": h, "cached": cached})
    all_refs = ([ref_tensor] if ref_tensor is not None else []) + extra_tensors
    print(f"  output {w}x{h}"
          + "".join(f", picture {n} {t.shape[2]}x{t.shape[1]}" for n, t in enumerate(all_refs, 1))
          + f", {'reusing cached prompt/image encoding' if cached else 'encoding prompt/image'}")

    seed = seed if seed and seed > 0 else random.randint(1, 2**32 - 1)
    outputs, seeds = [], []
    with torch.inference_mode():
        if cached:
            pos, neg, latent = _COND_CACHE.pop(key)
        else:
            latent = None
            if ref_tensor is not None:
                ref_latent = VAEEncode.encode(vae, ref_tensor)[0]
                images, refs = [ref_tensor], [ref_latent["samples"]]
                if (ref_tensor.shape[2], ref_tensor.shape[1]) == (w, h):
                    latent = ref_latent  # same frame: reuse, no second VAE encode
                elif partial:
                    latent = VAEEncode.encode(vae, pil_to_comfy(center_crop_resize(input_pil, w, h)))[0]
            else:
                images, refs = [], []
            for t in extra_tensors:
                images.append(t)
                refs.append(VAEEncode.encode(vae, t)[0]["samples"])
            if latent is None:
                latent = {"samples": torch.zeros([1, 16, h // 8, w // 8])}
            pos = qwen_edit_encode(clip, positive, vae=vae, images=images, ref_latents=refs)
            if use_neg:
                neg = qwen_edit_encode(clip, negative, vae=vae, images=images, ref_latents=refs)
            else:
                neg = ConditioningZeroOut.zero_out(pos)[0]
        _COND_CACHE[key] = (pos, neg, latent)  # (re)insert as most recent
        while len(_COND_CACHE) > _COND_CACHE_MAX:
            _COND_CACHE.pop(next(iter(_COND_CACHE)))

        for i in range(max(1, int(batch_size))):
            s = (seed + i) % (2**32)
            samples = KSampler.sample(model or unet, s, steps, cfg, sampler, scheduler, pos, neg, latent, denoise)[0]
            img = comfy_to_pil(VAEDecode.decode(vae, samples)[0])
            outputs.append(img)
            seeds.append(s)
            if on_image:
                on_image(i, img)
    return outputs, seeds


print(f"{MODEL_NAME} loaded successfully.")

/content/ComfyUI
Qwen-Rapid-NSFW-v23_Q2_K.gguf loaded successfully.


In [ ]:
# Cell 3 (optional): quick test edit inside the notebook
%cd /content/ComfyUI

import os
from PIL import Image
from IPython.display import display

if "run_qwen_edit" not in globals():
    raise RuntimeError("Run cell 2 (model loader) first.")

# Upload your own image to /content/input.png, otherwise a flat placeholder is used
INPUT_PATH = "/content/input.png"
if os.path.exists(INPUT_PATH):
    input_pil = Image.open(INPUT_PATH).convert("RGB")
else:
    input_pil = Image.new("RGB", (768, 768), (120, 140, 160))

positive = "transform the scene into a dramatic studio portrait with rim lighting"
negative = ""

torch.cuda.empty_cache()
images, seeds = run_qwen_edit(positive, negative, input_pil=input_pil, width=768, height=768, **{
    k: AIO_DEFAULTS[k] for k in ("steps", "cfg", "sampler", "scheduler", "denoise")
})
print("seed:", seeds[0])
display(images[0])

In [ ]:
# Cell 4: API server for the web frontend (Cloudflare tunnel)
# Finishes after printing the READY: URL; the server keeps running in the background, so you can run
# other cells meanwhile. Re-running this cell updates the running server instead of starting a new one.
%cd /content/ComfyUI

import os
import io
import re
import gc
import time
import uuid
import base64
import threading
import subprocess
import traceback
import requests

import urllib.parse

import torch
import comfy.utils
from nodes import NODE_CLASS_MAPPINGS
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio

nest_asyncio.apply()

if "run_qwen_edit" not in globals():
    raise RuntimeError("Models are not loaded in this runtime. Run cell 2 first, then this cell.")

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ---------------- LoRAs ----------------
# Qwen-Image / Qwen-Image-Edit LoRAs (not SDXL ones). Download them from the website with
# "+ Download LoRA", or put .safetensors files in /content/ComfyUI/models/loras yourself.
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")
LORA_DIR = "/content/ComfyUI/models/loras"
CKPT_DIR = LORA_DIR  # downloads from this backend are always LoRAs
os.makedirs(LORA_DIR, exist_ok=True)
MODEL_EXTS = (".safetensors", ".ckpt", ".pt", ".pth", ".bin")

# ---------------- LoRA loading with a compatibility check ----------------
# ComfyUI silently skips LoRA layers that don't exist in the model, so a LoRA made for another
# model (e.g. SDXL on Qwen) would quietly do nothing. Count what matches and say so.
import comfy.lora
import comfy.lora_convert
import comfy.sd
import folder_paths

_LORA_SUFFIXES = (".lora_up.weight", ".lora_down.weight", ".lora_A.weight", ".lora_B.weight",
                  ".lora_mid.weight", ".lora.up.weight", ".lora.down.weight", ".diff", ".hada_w1_a",
                  ".lokr_w1", ".lokr_w1_a", ".oft_blocks", ".w_norm", ".set_weight")


def apply_lora_checked(model, clip, name, strength_model, strength_clip, model_label):
    """Apply a LoRA like ComfyUI's LoraLoader. Returns (model, clip, warning_or_None).
    Raises ValueError if none of the LoRA's layers match the model."""
    path = folder_paths.get_full_path_or_raise("loras", name)
    lora, metadata = comfy.utils.load_torch_file(path, safe_load=True, return_metadata=True)
    key_map = {}
    if model is not None:
        key_map = comfy.lora.model_lora_keys_unet(model.model, key_map)
    if clip is not None:
        key_map = comfy.lora.model_lora_keys_clip(clip.cond_stage_model, key_map)
    converted = comfy.lora_convert.convert_lora(dict(lora))
    patches = comfy.lora.load_lora(converted, key_map, log_missing=False)
    matched = sum(1 for k in patches if not (isinstance(k, str) and k.endswith(".config")))
    layers = {k[: -len(s)] for k in converted for s in _LORA_SUFFIXES if k.endswith(s)}
    total = len(layers)
    if matched == 0:
        raise ValueError(
            f"LoRA '{name}' doesn't match the {model_label} model (0 of {total or '?'} layers matched). "
            f"It was probably made for a different model — remove it or pick a {model_label} LoRA.")
    warning = None
    if total and matched < 0.5 * total:
        warning = (f"LoRA '{name}': only {matched} of {total} layers matched the {model_label} model, "
                   f"so it may have little effect (probably made for a different model version).")
    print(f"  LoRA {name} @ {strength_model}: {matched}/{total or '?'} layers matched")
    new_model, new_clip = comfy.sd.load_lora_for_models(model, clip, lora, strength_model, strength_clip,
                                                        lora_metadata=metadata)
    return new_model, new_clip, warning


def list_files(folder):
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(MODEL_EXTS):
                out.append(os.path.relpath(os.path.join(root, f), folder))
    return sorted(out, key=str.lower)



_lora_model_cache = {"key": None, "model": None, "warnings": []}


def model_with_loras(loras):
    """Base GGUF model with the requested LoRAs applied (patches are applied on the fly).
    The last combination is kept, so repeat generations don't re-read the LoRA files.
    Returns (model, [(name, strength)], [warnings])."""
    wanted = []
    for l in loras or []:
        name = (l.get("name") if isinstance(l, dict) else str(l)) or ""
        strength = float(l.get("strength", 1.0)) if isinstance(l, dict) else 1.0
        if name and strength != 0:
            wanted.append((name, strength))
    if not wanted:
        return unet, [], []
    key = tuple(wanted)
    if _lora_model_cache["key"] == key:
        return _lora_model_cache["model"], wanted, _lora_model_cache["warnings"]
    available = set(list_files(LORA_DIR))
    model, warnings = unet, []
    with torch.inference_mode():
        for name, strength in wanted:
            if name not in available:
                raise ValueError(f"LoRA not found on the server: {name}")
            model, _, warning = apply_lora_checked(model, None, name, strength, 0, "Qwen-Image-Edit")
            if warning:
                warnings.append(warning)
    _lora_model_cache.update(key=key, model=model, warnings=warnings)
    return model, wanted, warnings


# ---------------- LoRA downloads (background job, streamed to disk) ----------------
def resolve_url(url):
    """Turn page links into direct download links. Returns (download_url, suggested_name)."""
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    token = (qs.get("token") or [CIVITAI_TOKEN])[0]
    host = parsed.netloc.lower()

    if "civitai.com" in host or "civitai.red" in host:
        api_headers = {"Authorization": f"Bearer {token}"} if token else {}
        m = re.search(r"/api/download/models/(\d+)", parsed.path)
        if not m:
            version_id = (qs.get("modelVersionId") or [None])[0]
            if not version_id:
                mm = re.search(r"/models/(\d+)", parsed.path)
                if not mm:
                    raise ValueError("Unrecognised Civitai link. Use a model page or download link.")
                info = requests.get(f"https://civitai.com/api/v1/models/{mm.group(1)}", headers=api_headers, timeout=30)
                info.raise_for_status()
                versions = info.json().get("modelVersions") or []
                if not versions:
                    raise ValueError("That Civitai model has no downloadable versions.")
                version_id = versions[0]["id"]  # newest version
            url = f"https://civitai.com/api/download/models/{version_id}"
        else:
            url = f"https://civitai.com/api/download/models/{m.group(1)}"
        if token:
            url += "?token=" + urllib.parse.quote(token)
        return url, None

    if "huggingface.co" in host:
        url = url.replace("/blob/", "/resolve/")
        return url, os.path.basename(urllib.parse.urlparse(url).path)

    return url, os.path.basename(parsed.path)


def filename_from_response(resp, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
    name = urllib.parse.unquote(m.group(1).strip()) if m else (fallback or "")
    name = os.path.basename(name).strip().strip('"')
    name = re.sub(r"[^\w.\-+() ]", "_", name)
    return name


def run_download(job_id, url, kind):
    job = JOBS[job_id]
    job["status"] = "running"
    folder = LORA_DIR if kind == "lora" else CKPT_DIR
    tmp_path = None
    try:
        dl_url, suggested = resolve_url(url)
        with requests.get(dl_url, stream=True, timeout=60, allow_redirects=True,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            if r.status_code in (401, 403):
                raise ValueError("Access denied — this model needs a Civitai API key (set CIVITAI_TOKEN or add ?token=...)")
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError("The link returned a web page, not a model file. Use the download link.")
            name = filename_from_response(r, suggested)
            if not name.lower().endswith(MODEL_EXTS):
                name = f"{kind}_{int(time.time())}.safetensors"
            job["filename"] = name
            path = os.path.join(folder, name)
            if os.path.exists(path):
                job["status"], job["progress"] = "done", 100
                return
            total = int(r.headers.get("content-length") or 0)
            job["total_mb"] = round(total / 1e6) if total else None
            tmp_path = path + ".part"
            done = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
                    done += len(chunk)
                    job["done_mb"] = round(done / 1e6)
                    job["progress"] = round(done * 100 / total) if total else None
            os.replace(tmp_path, path)
            tmp_path = None
        job["status"], job["progress"] = "done", 100
        print(f"Downloaded {kind}: {name} ({job.get('done_mb')} MB)")
    except Exception as e:
        traceback.print_exc()
        job["status"], job["error"] = "error", str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)



class GenerateRequest(BaseModel):
    positive: str = ""
    negative: str = ""
    steps: int = AIO_DEFAULTS["steps"]
    cfg: float = AIO_DEFAULTS["cfg"]
    width: int = AIO_DEFAULTS["width"]
    height: int = AIO_DEFAULTS["height"]
    seed: int = -1
    sampler: str = AIO_DEFAULTS["sampler"]
    scheduler: str = AIO_DEFAULTS["scheduler"]
    denoise: float = AIO_DEFAULTS["denoise"]
    batch_size: int = 1
    checkpoint: str | None = None
    loras: list = []
    reference_image: str | None = None
    reference_images: list = []  # Pictures 2 and 3 (reference_image is Picture 1)


def decode_base64_image(b64_str):
    """Accepts base64 / data: URLs, or an http(s) image address (downloaded here, since
    browsers block most cross-site image downloads)."""
    if not b64_str:
        return None
    value = b64_str.strip()
    if re.match(r"^https?://", value, re.I):
        r = requests.get(value, timeout=30, headers={
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
            "Accept": "image/avif,image/webp,image/png,image/jpeg,image/*;q=0.8,*/*;q=0.5",
        })
        r.raise_for_status()
        try:
            return Image.open(io.BytesIO(r.content)).convert("RGB")
        except Exception:
            raise ValueError(f"The address did not return an image (content-type: {r.headers.get('content-type')})")
    cleaned = re.sub(r"^data:image/[\w.+-]+;base64,", "", value)
    return Image.open(io.BytesIO(base64.b64decode(cleaned))).convert("RGB")


def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# Generation runs in a background job. Cloudflare quick tunnels drop requests that take
# longer than ~100 s, which a T4 easily exceeds, so the frontend polls /job/{id} instead.
# Kept across re-runs of this cell, so jobs that are running keep their state and never run two at once
JOBS = globals().get("JOBS") if isinstance(globals().get("JOBS"), dict) else {}
gpu_lock = globals().get("gpu_lock") if isinstance(globals().get("gpu_lock"), type(threading.Lock())) else threading.Lock()
current_job = globals().get("current_job") if isinstance(globals().get("current_job"), dict) else {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def new_job(kind, **extra):
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"kind": kind, "status": "queued", "created": now, "error": None, **extra}
    return job_id


def memory_report():
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = f"RAM {(vm.total - vm.available) / 1e9:.1f}/{vm.total / 1e9:.1f} GB"
    except Exception:
        ram = "RAM ?"
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        return f"{ram}, VRAM {(total - free) / 1e9:.1f}/{total / 1e9:.1f} GB"
    return ram


def run_job(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        current_job["id"] = job_id
        job["status"] = "running"
        try:
            batch = max(1, min(int(req.batch_size or 1), 8))
            steps = max(1, min(int(req.steps), 50))
            input_pil = decode_base64_image(req.reference_image)
            extra_pils = [decode_base64_image(x) for x in (req.reference_images or [])[:2] if x]
            if input_pil is None and extra_pils:
                input_pil = extra_pils.pop(0)  # the first image given is always Picture 1
            model, used_loras, lora_warnings = model_with_loras(req.loras)
            job["warnings"] = lora_warnings
            n_pics = (1 if input_pil else 0) + len(extra_pils)
            print(f"[{job_id}] {'edit' if n_pics else 'txt2img'}{f' with {n_pics} pictures' if n_pics > 1 else ''}: {steps} steps, cfg={req.cfg}, "
                  f"{req.sampler}/{req.scheduler}, batch={batch}"
                  + (f", LoRAs: {', '.join(f'{n} @ {w}' for n, w in used_loras)}" if used_loras else ""))

            gc.collect()
            torch.cuda.empty_cache()

            def on_info(info):
                job["width"], job["height"], job["cached"] = info["width"], info["height"], info["cached"]

            def on_image(i, img):
                job["images"].append(pil_to_b64(img))
                job["done"] = i + 1

            _, seeds = run_qwen_edit(
                req.positive, req.negative, input_pil=input_pil,
                width=req.width, height=req.height, steps=steps, cfg=req.cfg,
                sampler=req.sampler, scheduler=req.scheduler, seed=req.seed,
                denoise=req.denoise, batch_size=batch, on_image=on_image, on_info=on_info,
                model=model, extra_images=extra_pils,
            )
            job["seeds"] = seeds
            job["status"] = "done"
            print(f"[{job_id}] done")
        except Exception as e:
            traceback.print_exc()
            job["status"] = "error"
            job["error"] = str(e)
        finally:
            current_job["id"] = None
            gc.collect()
            torch.cuda.empty_cache()
            print(f"[{job_id}] memory after job: {memory_report()}")


def job_payload(job_id):
    job = JOBS[job_id]
    if job.get("kind") == "download":
        return {"job_id": job_id, **{k: v for k, v in job.items() if k != "created"}}
    out = {
        "job_id": job_id,
        "status": job["status"],
        "batch_size": job["batch_size"],
        "done": job["done"],
        "step": job["step"],
        "total_steps": job["total_steps"],
        "width": job.get("width"),
        "height": job.get("height"),
        "cached": job.get("cached", False),
        "warnings": job.get("warnings", []),
    }
    if job["status"] == "done":
        out["images"] = job["images"]
        out["image"] = job["images"][0] if job["images"] else None
        out["seeds"] = job["seeds"]
        out["seed"] = job["seeds"][0] if job["seeds"] else None
    if job["status"] == "error":
        out["error"] = job["error"]
    return out


@app.get("/")
def root():
    return {"status": "ok", "model": MODEL_NAME, "mode": "qwen_image_edit"}


@app.get("/user_models")
def get_user_models():
    return {
        "checkpoints": [MODEL_NAME],
        "lora_files": list_files(LORA_DIR),
        "supports_loras": True,
        "max_reference_images": 3,
        "model_type": "qwen_image_edit",
        "defaults": AIO_DEFAULTS,
        "samplers": ["sa_solver", "euler", "euler_ancestral", "res_multistep", "er_sde", "dpmpp_2m"],
        "schedulers": ["beta", "simple", "normal", "sgm_uniform", "karras"],
    }


@app.post("/generate")
def generate(req: GenerateRequest):
    # Drop finished jobs older than 30 min
    job_id = new_job(
        "generate", images=[], seeds=[],
        batch_size=max(1, min(int(req.batch_size or 1), 8)), done=0, step=0, total_steps=0,
    )
    threading.Thread(target=run_job, args=(job_id, req), daemon=True).start()
    return job_payload(job_id)


class ModelDownload(BaseModel):
    url: str
    type: str = "lora"


@app.post("/download_model")
def download_model(req: ModelDownload):
    if req.type != "lora":
        return {"status": "error", "error": "This Qwen backend only downloads LoRAs (the model is fixed)."}
    job_id = new_job("download", type="lora", filename=None, progress=0, done_mb=0, total_mb=None)
    threading.Thread(target=run_download, args=(job_id, req.url, "lora"), daemon=True).start()
    return job_payload(job_id)


@app.get("/job/{job_id}")
def get_job(job_id: str):
    if job_id not in JOBS:
        return {"status": "error", "error": "Unknown job (the server may have restarted)"}
    return job_payload(job_id)


async def serve_latest_app(scope, receive, send):
    # Always hand requests to the current `app`, so re-running this cell updates the running server
    await globals()["app"](scope, receive, send)


if globals().get("QWEN_SERVER_URL"):
    print("=" * 50)
    print(f"Server updated and still running. READY: {QWEN_SERVER_URL}")
    print("=" * 50)
else:
    # Don't kill port 8000 here: the server runs inside this notebook, so that would crash it.
    !pkill -f cloudflared >/dev/null 2>&1 || true

    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("Installing Cloudflare tunnel...")
        !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x /usr/local/bin/cloudflared

    threading.Thread(target=lambda: uvicorn.run(serve_latest_app, host="0.0.0.0", port=8000, log_level="warning"),
                     daemon=True).start()
    time.sleep(4)

    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print("Starting tunnel, waiting for address...")
    url = None
    start_time = time.time()
    while time.time() - start_time < 60:
        line = cf_proc.stdout.readline()
        if not line:
            break
        match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break

    # Keep draining cloudflared output so its pipe never fills up and blocks the tunnel
    threading.Thread(target=lambda: [None for _ in cf_proc.stdout], daemon=True).start()

    if url:
        QWEN_SERVER_URL = url
        print("=" * 50)
        print(f"READY: {url}")
        print("Paste this URL into the frontend's Backend URL field.")
        print("This cell is done; the server keeps running. Job logs appear below.")
        print("To stop the server: Runtime → Restart session.")
        print("=" * 50)
    else:
        print("Failed to retrieve Cloudflare tunnel URL. Run this cell again.")
